In [1]:
import requests
import json
from pprint import pprint
import os
from dotenv import load_dotenv
from pprint import pprint

# Load environment variables from repo root
load_dotenv("../.env")

api_key = os.getenv("SURF_API_KEY")
willma_base_url = os.getenv("SURF_API_BASE", "https://willma.surf.nl/api/v0")

if not api_key:
    raise ValueError("SURF_API_KEY is not set in .env")

headers = {"X-API-KEY": api_key, "Content-Type": "application/json"}

models = requests.get(f"{willma_base_url}/sequences", headers=headers, timeout=30).json()
text_models = [m["name"] for m in models if m.get("sequence_type") == "text"]

print("Available text models:", text_models)
model_name = "default-text-large" if "default-text-large" in text_models else (text_models[0] if text_models else None)
if not model_name:
    raise RuntimeError("No text models returned by /sequences")
pprint(text_models)


Available text models: ['default-text-large', 'openai/gpt-oss-120b', 'Qwen/Qwen2.5-Coder-32B-Instruct-AWQ', 'Qwen/Qwen2.5-VL-32B-Instruct-AWQ']
['default-text-large',
 'openai/gpt-oss-120b',
 'Qwen/Qwen2.5-Coder-32B-Instruct-AWQ',
 'Qwen/Qwen2.5-VL-32B-Instruct-AWQ']


In [2]:
response = requests.post(
    f"{willma_base_url}/chat/completions",
    data=json.dumps({
        "model": "Qwen/Qwen2.5-VL-32B-Instruct-AWQ",
        "messages": [{"role": "user", "content": "Hoi!"}],
    }),
    headers=headers,
    timeout=60,
).json()


In [3]:
response

{'id': 'chatcmpl-trace_aa941e5e-bf10-48f8-ae55-3411adceb057',
 'created': 1782730489,
 'model': 'Qwen/Qwen2.5-VL-32B-Instruct-AWQ',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Hello! How can I assist you today? 😊'},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 22, 'completion_tokens': 12, 'total_tokens': 34},
 'surf': {'gpu_power': 0.0, 'compute_time_s': 0.203309}}

The selected model is discovered from `/sequences` and stored in `model_name`.

If you want a fixed model, set `model_name` manually after running the first cell.
